## Загрузка данных и создание датафрейма

In [12]:
import pandas as pd

views = pd.read_csv("data/feed-views.log", sep='\t',engine='python',names=['datetime','user'])

views.head()

,datetime,user
0,2020-04-17 12:01:08.463179,artem
1,2020-04-17 12:01:23.743946,artem
2,2020-04-17 12:27:30.646665,artem
3,2020-04-17 12:35:44.884757,artem
4,2020-04-17 12:35:52.735016,artem


## Преобразование datetime и извлечение компонентов времени

In [13]:
views['datetime'] = pd.to_datetime(views['datetime'], errors='coerce')
views['year'] = views['datetime'].dt.year
views['month'] = views['datetime'].dt.month
views['day'] = views['datetime'].dt.day
views['hour'] = views['datetime'].dt.hour
views['minute'] = views['datetime'].dt.minute
views['second'] = views['datetime'].dt.second
views.head()

,datetime,user,year,month,day,hour,minute,second
0,2020-04-17 12:01:08.463179,artem,2020,4,17,12,1,8
1,2020-04-17 12:01:23.743946,artem,2020,4,17,12,1,23
2,2020-04-17 12:27:30.646665,artem,2020,4,17,12,27,30
3,2020-04-17 12:35:44.884757,artem,2020,4,17,12,35,44
4,2020-04-17 12:35:52.735016,artem,2020,4,17,12,35,52


## Создание столбца daytime с помощью метода cut и установка пользователя в качестве индекса

In [14]:
bins = [0, 4, 7, 11, 17, 20, 24]
labels = ['night', 'early morning', 'morning', 'afternoon', 'early evening', 'evening']

views['daytime'] = pd.cut(views['hour'], bins=bins, labels=labels, right=False, include_lowest=True)

views = views.set_index('user')
views.head()

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
artem,2020-04-17 12:01:08.463179,2020,4,17,12,1,8,afternoon
artem,2020-04-17 12:01:23.743946,2020,4,17,12,1,23,afternoon
artem,2020-04-17 12:27:30.646665,2020,4,17,12,27,30,afternoon
artem,2020-04-17 12:35:44.884757,2020,4,17,12,35,44,afternoon
artem,2020-04-17 12:35:52.735016,2020,4,17,12,35,52,afternoon


## Расчет общего количества записей и подсчет количества записей по времени дня

In [15]:
print(f'Общее количество записей: {views['datetime'].count()}\
      \nКол-во записей по времени суток:\n{views['daytime'].value_counts()}')

Общее количество записей: 1076      
Кол-во записей по времени суток:
daytime
evening          509
afternoon        252
early evening    145
night            129
morning           36
early morning      5
Name: count, dtype: int64


## Сортировка значений по времени

In [16]:
views = views.sort_values(by=['hour', 'minute', 'second'], ascending=True)
views.head()

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
valentina,2020-05-15 00:00:13.222265,2020,5,15,0,0,13,night
valentina,2020-05-15 00:01:05.153738,2020,5,15,0,1,5,night
pavel,2020-05-12 00:01:27.764025,2020,5,12,0,1,27,night
pavel,2020-05-12 00:01:38.444917,2020,5,12,0,1,38,night
pavel,2020-05-12 00:01:55.395042,2020,5,12,0,1,55,night


## Расчет наибольшего часа для ночи, наименьшего для утра (с примерами пользователей) и моды (час и время суток)

In [17]:
night_max_hour = views.loc[views['daytime'] == 'night', 'hour'].max()
night_example_user = views.loc[views['hour'] == night_max_hour].index[0]

morning_min_hour = views.loc[views['daytime'] == 'morning', 'hour'].min()
morning_example_user = views.loc[views['hour'] == morning_min_hour].index[0]

hour_mode = views['hour'].mode().iloc[0]
daytime_mode = views['daytime'].mode().iloc[0]

print(f'Максимальный час для ночи: {night_max_hour}, пример пользователя: {night_example_user}')
print(f'Минимальный час для утра: {morning_min_hour}, пример пользователя: {morning_example_user}')
print(f'Наиболее частый час в записях: {hour_mode}\nНаиболее частое время суток: {daytime_mode}')

Максимальный час для ночи: 3, пример пользователя: konstantin
Минимальный час для утра: 8, пример пользователя: alexander
Наиболее частый час в записях: 22
Наиболее частое время суток: evening


## Показать самые ранний и поздний часы и пользователей

In [18]:
print(f'Самый ранний час и пользователь(-и):\n{views.nsmallest(3, 'hour')['hour']}')
print(f'Самый поздний час и пользователь(-и):\n{views.nlargest(3, 'hour')['hour']}')

Самый ранний час и пользователь(-и):
user
valentina    0
valentina    0
pavel        0
Name: hour, dtype: int32
Самый поздний час и пользователь(-и):
user
ekaterina    23
ekaterina    23
ekaterina    23
Name: hour, dtype: int32


## Использование метода describe и расчет IQR

In [19]:
stats = views.describe()
iqr = stats.loc['75%', 'hour'] - stats.loc['25%', 'hour']
print(iqr)

9.0
